In [1]:
import plotly.express as px
import pandas as pd
from bike_demand_forecasting.utils import get_paths

## **1. EXTRACT DATA**

In [5]:
from bike_demand_forecasting.io import extract_all_zips

In [3]:
# Instanciate Path function
paths = get_paths()

# Retrieve Paths
DATA_DIR_2024 = paths["DATA_DIR_2024"]
DATA_EXTRACTED_DIR_2024 = paths["DATA_EXTRACTED_DIR_2024"]

DATA_DIR_2025 = paths["DATA_DIR_2025"]
DATA_EXTRACTED_DIR_2025 = paths["DATA_EXTRACTED_DIR_2025"]

# Instanciate extrect function 2024
extract_all_zips(DATA_DIR_2024, DATA_EXTRACTED_DIR_2024)

# Instanciate extract function for 2025
extract_all_zips(DATA_DIR_2025, DATA_EXTRACTED_DIR_2025)

NameError: name 'extract_all_zips' is not defined

## **2. MERGED DATA** ##

In [6]:
from bike_demand_forecasting.io import merge_all_csv

In [7]:
# Retrieve Path
DATA_DIR_2024_MERGED = paths["DATA_DIR_2024_MERGED"]
DATA_DIR_2025_MERGED = paths["DATA_DIR_2025_MERGED"]

# Instanciate merge function for 2024
merged_2024_path = merge_all_csv(DATA_EXTRACTED_DIR_2024, DATA_DIR_2024_MERGED)

# Instanciate merge function for 2025
merged_2025_path = merge_all_csv(DATA_EXTRACTED_DIR_2025, DATA_DIR_2025_MERGED)

Merged file created: /mnt/c/Users/33760/Desktop/bike-demand-forecasting/data/merged/2024.csv
Merged file created: /mnt/c/Users/33760/Desktop/bike-demand-forecasting/data/merged/2025.csv


In [8]:
# Read data
df_2024 = pd.read_csv(merged_2024_path)
df_2025 = pd.read_csv(merged_2025_path)

# Concat 2024 & 2025 data
df_merged = pd.concat([df_2024, df_2025], ignore_index=True)

/tmp/ipykernel_80187/1710811876.py:3: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2025 = pd.read_csv(merged_2025_path)


In [9]:
DATA_DIR_MERGED = paths["DATA_DIR_MERGED"]

# Read 2024-2025 merged data
data_merged_path = DATA_DIR_MERGED / "all_merged.csv"

# Save Dataframe
df_merged.to_csv(data_merged_path, index = False)

## **3. EDA**

In [ ]:
DATA_DIR_MERGED = paths["DATA_DIR_MERGED"]

# Define path
data_merged_path = DATA_DIR_MERGED / "all_merged.csv"

# Read CSV Data 2024 & 2025 concat
df = pd.read_csv(data_merged_path)

/tmp/ipykernel_910/1723419287.py:7: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_merged_path)


### **3. 1 processing for EDA**

In [ ]:
# View Columns
df_merged.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='object')

In [ ]:
# Missing values and sort
df_merged.isnull().mean().sort_values(ascending = False)

end_station_id        2.189706e-01
end_station_name      2.187042e-01
start_station_name    2.113889e-01
start_station_id      2.113889e-01
end_lat               7.159760e-04
end_lng               7.159760e-04
member_casual         7.826585e-08
ended_at              7.826585e-08
ride_id               7.826585e-08
rideable_type         0.000000e+00
started_at            0.000000e+00
start_lat             0.000000e+00
start_lng             0.000000e+00
dtype: float64

In [ ]:
# Data types
df_merged.dtypes

ride_id                object
rideable_type          object
started_at             object
ended_at               object
start_station_name     object
start_station_id      float64
end_station_name       object
end_station_id        float64
start_lat             float64
start_lng              object
end_lat               float64
end_lng               float64
member_casual          object
dtype: object

In [ ]:
df_merged.head()

In [14]:
# Test parsing on temporary variable

s = df_merged["started_at"].astype(str) 
# convert to panda Saries
s = pd.Series(s).str.strip() 
# convert to datetime
dt = pd.to_datetime(s, errors="coerce") 
print("NaT rate:", dt.isna().mean())
# Display fail paring examples
bad = s[dt.isna()].head(30)
print("Examples that fail parsing:\n", bad.to_string())

NaT rate: 0.8434583520780053
Examples that fail parsing:
 2000127    2024-06-16 09:21:19.510
2000128    2024-06-09 08:49:48.266
2000129    2024-06-16 10:46:28.432
2000130    2024-06-08 11:54:19.244
2000131    2024-06-15 13:33:26.602
2000132    2024-06-26 21:40:49.395
2000133    2024-06-11 19:10:27.907
2000134    2024-06-12 17:58:57.070
2000135    2024-06-01 01:50:56.012
2000136    2024-06-17 08:13:00.021
2000137    2024-06-28 19:23:33.107
2000138    2024-06-29 20:37:05.317
2000139    2024-06-04 09:11:44.609
2000140    2024-06-24 20:06:32.993
2000141    2024-06-01 14:12:40.383
2000142    2024-06-05 18:21:52.529
2000143    2024-06-13 14:49:11.664
2000144    2024-06-15 14:46:40.213
2000145    2024-06-13 17:00:43.721
2000146    2024-06-05 19:31:59.881
2000147    2024-06-27 22:16:22.517
2000148    2024-06-08 01:39:27.126
2000149    2024-06-21 19:59:17.780
2000150    2024-06-07 17:21:24.473
2000151    2024-06-15 22:01:29.278
2000152    2024-06-08 16:13:46.795
2000153    2024-06-23 08:19:35.7

!!! Millisecond format create NaN values with parsing !!! We need to hundle this

In [15]:
# Harmonize started_at colums (space, type)
df_merged["started_at"] = df_merged["started_at"].astype("string").str.strip()

# Erase ".xxx" (millisecond) from "started_at" column
df_merged["started_at"] = df_merged["started_at"].str.replace(r"\.\d+$", "", regex=True)

# Convert "started_at" column to datetime
df_merged["started_at"] = pd.to_datetime(df_merged["started_at"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

print("NaT rate:", df_merged["started_at"].isna().mean())

NaT rate: 7.826585407926327e-08


In [16]:
# Keep only the started_at column since we'll change the data granularity
# by aggregating into 1-hour windows.
df_started = df_merged[["started_at"]].copy()

In [17]:
# Build hourly target y (global demand), including hours with 0 rides
# Avoid resampling 
dt = df_started["started_at"]
# Number of rides per hour
y = dt.dt.floor("H").value_counts()
# Create a continuous hourly grid to avoid missing hour (row without bike ride)
full_hours = pd.date_range(dt.min().floor("H"), dt.max().floor("H"), freq="H")

# reindex on the full grid
df_started_hourly_full = (
    y.reindex(full_hours, fill_value=0)
     .sort_index() # hour sort (ascending)
     .rename("y") # column name
     .reset_index() # index to column
     .rename(columns={"index": "started_at"})
)

/tmp/ipykernel_80187/1944332252.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  y = dt.dt.floor("H").value_counts()
/tmp/ipykernel_80187/1944332252.py:7: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_hours = pd.date_range(dt.min().floor("H"), dt.max().floor("H"), freq="H")


In [18]:
# Rows add with zero rides target (y)
print(f"df_started_hourly: {len(y)} rows")
print(f"df_started_hourly_full: {len(df_started_hourly_full)} rows")
print(f"{len(df_started_hourly_full) - len(y)} hourly time slots (2024–2025) with zero rides")

df_started_hourly: 17542 rows
df_started_hourly_full: 17544 rows
2 hourly time slots (2024–2025) with zero rides


In [19]:
# Temporal features
d = df_started_hourly_full["started_at"]
df_started_hourly_full["year"] = d.dt.year
df_started_hourly_full["month"] = d.dt.month
df_started_hourly_full["day"] = d.dt.day
df_started_hourly_full["hour"] = d.dt.hour
df_started_hourly_full["dayofweek"] = d.dt.dayofweek
df_started_hourly_full["is_weekend"] = (df_started_hourly_full["dayofweek"] >= 5).astype("int8")
df_started_hourly_full["dayofyear"] = d.dt.dayofyear

In [20]:
# Sort df by date
df_started_hourly_full = df_started_hourly_full.sort_values(by = ["started_at"], ascending = True)

In [21]:
DATA_DIR = paths["DATA_DIR"]
DATA_PROCESSED_DIR = DATA_DIR / "processed"

# Create folder
DATA_PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

# Define Path
data_started_hourly_full_path = DATA_PROCESSED_DIR / "df_started_hourly_full.csv"

# Save
df_started_hourly_full.to_csv(data_started_hourly_full_path, index = False)

### **3. 2 Plots at different temporal resolutions**

In [22]:
# Read data
df_started_hourly_full = pd.read_csv(data_started_hourly_full_path)

In [23]:
# Display data
df_started_hourly_full.head()

,started_at,y,year,month,day,hour,dayofweek,is_weekend,dayofyear
0,2024-01-01 00:00:00,272,2024,1,1,0,0,0,1
1,2024-01-01 01:00:00,324,2024,1,1,1,0,0,1
2,2024-01-01 02:00:00,237,2024,1,1,2,0,0,1
3,2024-01-01 03:00:00,60,2024,1,1,3,0,0,1
4,2024-01-01 04:00:00,26,2024,1,1,4,0,0,1


In [24]:
fig = px.line(
    df_started_hourly_full,
    x="started_at",
    y="y",
    title="Bike demand (rides started) per hour"
)
fig.show()

In [25]:
# daily total rides (sum of hourly y)
df_daily = (
    df_started_hourly_full
    .groupby(["year", "dayofyear"], as_index=False)["y"]
    .sum()
)

fig = px.line(
    df_daily,
    x="dayofyear",
    y="y",
    color="year",
    title="Daily bike demand (sum of hourly rides) — 2024 vs 2025",
    labels={"dayofyear": "Day of year", "y": "Rides per day", "year": "Year"}
)
fig.show()

In [26]:
import plotly.express as px

# 1) daily total rides (sum of hourly y)
df_daily = (
    df_started_hourly_full
    .groupby(["year", "month", "day", "dayofweek"], as_index=False)["y"]
    .sum()
)

# 2) average daily demand by day of week
dow = (
    df_daily
    .groupby(["year", "dayofweek"], as_index=False)["y"]
    .mean()
)

# labels
dow_labels = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}
dow["dow"] = dow["dayofweek"].map(dow_labels)

fig = px.line(
    dow.sort_values("dayofweek"),
    x="dow",
    y="y",
    color="year",
    markers=True,
    title="Average daily demand by day of week - 2024 vs 2025",
    labels={"dow": "Day of week", "y": "Avg rides per day", "year": "Year"}
)
fig.show()

In [27]:
# 1) daily demand (sum of hourly y)
df_daily = (
    df_started_hourly_full
    .groupby(["year", "month", "day"], as_index=False)["y"]
    .sum()
)

# 2) build date + year-month label
df_daily["date"] = pd.to_datetime(dict(year=df_daily["year"], month=df_daily["month"], day=df_daily["day"]))
df_daily["year_month"] = df_daily["date"].dt.to_period("M").astype(str)  # "2024-01", ...

# 3) plotly box plot
fig = px.box(
    df_daily.sort_values("year_month"),
    x="year_month",
    y="y",
    color="year_month",          
    points="outliers",           
    title="Daily bike demand distribution by month (2024–2025)",
    labels={"year_month": "", "y": "Rides per day"}
)

fig.update_layout(
    xaxis_tickangle=-90,
    showlegend=False
)

fig.show()
